# V2 confirm run v2 — multi-seed Phase 1 + 2 + widened H6

Fixes from two review rounds:
1. **`intervention_only`** = top-k by `|U_j|` (no `R_j`) — Phase 2 tests whether the `R_j` gate adds invariance beyond dose-response alone.
2. **Multi-seed** (0/1/2) on ETTh1 + ETTh2 + electricity — error bars on `I(MVDF_v2)` vs `I(correlated)`.
3. **Invariance degeneracy guard** — a condition whose feature signal populates <3 quantile bins now yields `I = NaN` (was `~0`, which spuriously read as 'perfectly invariant' — this inflated the ETTh1 seed sd).
4. **H6 widened to a four-way decomposition** with a proper split-conformal calibration (50% MLP-train / 30% conformalize / 20% ID-width eval of `E_eval_ID`), so ID coverage is ~1-α by construction:
   - `dense` (FFDCI as-is) vs `mvdf` vs `dense+mvdf`
   - **`correlated`** — top-k correlation-selected features, same k, same scorer. *The sharpest test: does dose-response/`R_j` earn anything over the cheap baseline?*
   - **`random_k`** — k random SAE features (dimensionality control)
   - **`pca_k`** — top-k PCA of the dense vector (sensible-compression control)
   Read: `mvdf < dense` but `mvdf ≈ correlated` → any selection helps, not causal content. `mvdf < dense AND < correlated` → the `R_j` content matters. `mvdf ≈ random_k ≈ pca_k < dense` → dimensionality reduction alone.

**Guards:** data-fingerprint assert (channels exact for ETT/weather, ±1 for capped sets); per-dataset deterministic seeding.

Kaggle: GPU + Internet on, attach `uom220001c`. Run All. ~2-3 hr.

## 1. Vendor + deps + probe

In [ ]:
import os, base64, json, sys, subprocess, importlib
FILES=json.loads(base64.b64decode('...').decode())
for rel,c in FILES.items():
    p=os.path.join('mechanistic_conformal',rel); os.makedirs(os.path.dirname(p),exist_ok=True); open(p,'w').write(c)
print('wrote',len(FILES),'files')
for pkg in ['pyyaml','scipy','scikit-learn','statsmodels']:
    n={'pyyaml':'yaml','scikit-learn':'sklearn'}.get(pkg,pkg)
    try: importlib.import_module(n)
    except ImportError: subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=True)
import torch; print('torch',torch.__version__,'cuda',torch.cuda.is_available())
sys.path.insert(0,'mechanistic_conformal'); os.chdir('mechanistic_conformal')
from data.probe_inputs import probe; probe()


## 2. Set device + strict fingerprint on all configs

In [ ]:
import yaml, torch
dev='cuda' if torch.cuda.is_available() else 'cpu'
for cf in ['configs/v2_phase1.yaml','configs/v2_phase2.yaml','configs/v2_phase3_h6.yaml']:
    c=yaml.safe_load(open(cf)); c['device']=dev; c['strict_fingerprint']=True
    c['datasets']=["ETTh1", "ETTh2", "electricity", "traffic"]
    yaml.safe_dump(c,open(cf,'w'))
print('configs set: device',dev,'strict_fingerprint True, datasets',["ETTh1", "ETTh2", "electricity", "traffic"])


## 3. Phase 1 per seed  (dose-response, R_j, MVDF_v2, feature-type table)

In [ ]:
import runpy, sys, shutil
for sd in [0, 1, 2]:
    c=yaml.safe_load(open('configs/v2_phase1.yaml')); c['seed']=sd; c['run_name']=f'v2_phase1_seed{sd}'
    yaml.safe_dump(c,open('configs/_p1.yaml','w'))
    print('='*70,f'\n PHASE 1  seed {sd}\n','='*70)
    sys.argv=['p1','--config','configs/_p1.yaml','--taps','pre_head']
    try: runpy.run_module('experiments.phase1_core_mechanism', run_name='__main__')
    except SystemExit: pass
import pandas as pd, glob
for f in sorted(glob.glob('runs/v2_phase1_seed*/AGG_h1c.csv')):
    print(f); display(pd.read_csv(f))


## 4. Phase 2 per seed  +  aggregation diagnostic

**Phase 2's `I` is known-unreliable for `MVDF_v2` and `correlated`.** The bin-count audit showed their mean-aggregated feature signals populate only ~2-3 of 5 quantile bins (features fire on ~2% of windows by design — that's what makes them difficulty-*specific*). The aggregation diagnostic below tests whether max / L2 / active-count aggregation rescues the signal; the local run showed it does not — no aggregation of a ~2%-active 12-feature bundle varies enough. So H3 is reported as **not cleanly measurable, and the reason is structural** (the H1c selection criterion and the H3 metric pull against each other). H1c + H6 carry the paper; Phase 2 `I` is context, not a headline.

In [ ]:
import runpy, sys
for sd in [0, 1, 2]:
    c=yaml.safe_load(open('configs/v2_phase2.yaml')); c['seed']=sd; c['run_name']=f'v2_phase2_seed{sd}'
    yaml.safe_dump(c,open('configs/_p2.yaml','w'))
    print('='*70,f'\n PHASE 2  seed {sd}\n','='*70)
    sys.argv=['p2','--config','configs/_p2.yaml','--phase1_root',f'runs/v2_phase1_seed{sd}','--taps','pre_head']
    try: runpy.run_module('experiments.phase2_invariance', run_name='__main__')
    except SystemExit: pass

import pandas as pd, glob
frames=[]
for f in sorted(glob.glob('runs/v2_phase2_seed*/AGG_invariance.csv')):
    d=pd.read_csv(f); d['seed']=int(f.split('seed')[1].split('/')[0]); frames.append(d)
inv=pd.concat(frames, ignore_index=True)
inv.to_csv('runs/AGG_invariance_multiseed.csv', index=False)
deg_cols=[c for c in inv.columns if c.startswith('degenerate_')]
print('=== degenerate-I flags (min shared bins < 3) by condition ===')
if deg_cols: print(inv[deg_cols].sum().to_string())
print('\n=== I by condition across seeds (report WITH the degeneracy caveat) ===')
for ds,g in inv.groupby('dataset'):
    ic=g['I_correlated']; iv2=g['I_MVDF_v2']; iio=g['I_intervention_only']; iv1=g['I_MVDF_v1']
    print(f'{ds:12s} I_corr {ic.mean():.3f}+-{ic.std():.3f}  I_MVDFv2 {iv2.mean():.3f}+-{iv2.std():.3f}'
          f'  I_intOnly {iio.mean():.3f}+-{iio.std():.3f}  I_MVDFv1 {iv1.mean():.3f}+-{iv1.std():.3f}')

# aggregation diagnostic — does ANY aggregation give MVDF_v2/correlated a real signal?
sys.argv=['ad','--datasets']+[str(x) for x in ["ETTh1", "ETTh2", "electricity", "traffic"]]+['--seeds']+[str(x) for x in [0, 1, 2]]+['--phase1_root_tmpl','runs/v2_phase1_seed{seed}']
try: runpy.run_module('experiments.aggregation_diagnostic', run_name='__main__')
except SystemExit: pass
except Exception as e: print('agg diagnostic failed (non-fatal):', e)


## 5. Phase 3 — H6 four-way scorer decomposition (split-conformal)

In [ ]:
import runpy, sys
c=yaml.safe_load(open('configs/v2_phase3_h6.yaml')); c['device']=dev; c['strict_fingerprint']=True
c['datasets']=["ETTh1", "ETTh2", "electricity", "traffic"]; c['seeds']=[0, 1, 2]
yaml.safe_dump(c,open('configs/v2_phase3_h6.yaml','w'))
sys.argv=['p3','--config','configs/v2_phase3_h6.yaml','--phase1_root_tmpl','runs/v2_phase1_seed{seed}']
try: runpy.run_module('experiments.phase3_scorer_h6', run_name='__main__')
except SystemExit: pass
import pandas as pd
h6=pd.read_csv('runs/v2_phase3_h6/AGG_h6.csv')
print('=== coverage sanity (split-conformal -> should be ~0.90 ID) ===')
print(h6.groupby('mode')[['id_coverage','shift_coverage']].mean().round(3).to_string())
print('\n=== per-seed shift width by mode ===')
print(h6.pivot_table(index=['dataset','seed'],columns='mode',values='shift_width').round(3).to_string())
t=pd.read_csv('runs/v2_phase3_h6/AGG_h6_tests.csv')
print('\n=== paired bootstrap by comparison x env (mean over 9 dataset-seed runs) ===')
summ=t.groupby(['comparison','env']).agg(mean_diff=('mean_diff','mean'),p_a_lt_b=('p_a_lt_b','mean'),n_sig=('significant','sum'),n=('significant','size')).round(4)
print(summ.to_string())
print('\n=== FOUR-WAY READ ===')
def sig(comp,env): r=summ.loc[(comp,env)]; return f"{r.mean_diff:+.3f} (sig {int(r.n_sig)}/{int(r.n)})"
for env in ['ID','shift']:
    print(f'  [{env}] mvdf-dense {sig("mvdf_minus_dense",env)}  |  mvdf-correlated {sig("mvdf_minus_correlated",env)}  '
          f'|  mvdf-random_k {sig("mvdf_minus_random_k",env)}  |  mvdf-pca_k {sig("mvdf_minus_pca_k",env)}')
print('\n  mvdf<dense but mvdf~correlated       -> any sensible selection helps (not causal content)')
print('  mvdf<dense AND mvdf<correlated       -> dose-response/R_j content specifically matters')
print('  mvdf~random_k~pca_k, all < dense     -> dimensionality reduction alone explains it')


## 6. Zip

In [ ]:
import shutil; shutil.make_archive('/kaggle/working/v2_confirm','zip','runs')
print('runs -> /kaggle/working/v2_confirm.zip')
